In [14]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import rasterio
from PIL import Image
from skimage.transform import resize

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks


In [15]:
np.random.seed(42)
tf.random.set_seed(42)

## 2. Configuration
# Data directory where .npy files and captures JSON reside
data_dir = Path("./GHGsat")  # adjust as needed

# Resampling methods (match your built .npy suffixes)
resampling_methods = ["nearest", "bilinear", "bicubic", "lanczos"]

# Select method to train on
method = "bilinear"  # choose from resampling_methods

# Input shape
target_height, target_width = 128, 128
num_channels = 5  # CH4, CH4PL, CH4ER, ALB, FLG

In [16]:
## 3. Load Dataset
# Paths to .npy
X_train_path = data_dir / f"X_train_{method}.npy" if (data_dir / f"X_train_{method}.npy").exists() else data_dir / "X_train.npy"
X_test_path  = data_dir / f"X_test_{method}.npy"  if (data_dir / f"X_test_{method}.npy").exists()  else data_dir / "X_test.npy"
y_train_path = data_dir / f"y_train_{method}.npy" if (data_dir / f"y_train_{method}.npy").exists() else data_dir / "y_train.npy"
y_test_path  = data_dir / f"y_test_{method}.npy"  if (data_dir / f"y_test_{method}.npy").exists()  else data_dir / "y_test.npy"

# Load arrays
def load_data(path: Path):
    arr = np.load(path)
    print(f"Loaded {path.name}, shape={arr.shape}, dtype={arr.dtype}")
    return arr

X_train = load_data(X_train_path)
X_test  = load_data(X_test_path)
y_train = load_data(y_train_path)
y_test  = load_data(y_test_path)

assert X_train.ndim == 4 and X_train.shape[-1] == num_channels, "Unexpected channel dimension"



Loaded X_train.npy, shape=(241, 128, 128, 5), dtype=float32
Loaded X_test.npy, shape=(61, 128, 128, 5), dtype=float32
Loaded y_train.npy, shape=(241,), dtype=float64
Loaded y_test.npy, shape=(61,), dtype=float64


In [17]:
print("Train set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("Target range:", y_train.min(), "to", y_train.max())

# Check NaNs and value ranges
for arr, name in [(X_train, 'X_train'), (y_train, 'y_train')]:
    print(f"{name}: any NaN?", np.isnan(arr).any())
    if arr.ndim == 4:
        print(f"{name}: min={arr.min()}, max={arr.max()}")
    else:
        print(f"{name}: min={arr.min()}, max={arr.max()}")

## 5. Build Model
input_shape = (target_height, target_width, num_channels)

def build_model():
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),
        
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

model = build_model()
model.summary()

Train set size: (241, 128, 128, 5)
Test set size: (61, 128, 128, 5)
Target range: 136.0 to 7476.0
X_train: any NaN? False
X_train: min=0.0, max=5070.83642578125
y_train: any NaN? False
y_train: min=136.0, max=7476.0


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 128, 128, 32)   │         1,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 103,041 (402.50 KB)

 Trainable params: 102,593 (400.75 KB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
# Callbacks
early_stop = callbacks.EarlyStopping(patience=10, restore_best_weights=True)
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop]
)


In [ ]:
plt.figure(figsize=(8,4))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.title('Training vs Validation Loss')
plt.show()

## 8. Evaluation
# Predictions
y_pred = model.predict(X_test).flatten()

# R² score
r2 = r2_score(y_test, y_pred)
print(f"R² on test set: {r2:.3f}")

# Scatter plot
plt.figure(figsize=(6,6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('True Q-IME (kg/hr)')
plt.ylabel('Predicted Q-IME (kg/hr)')
plt.title(f'Predictions vs Truth (R²={r2:.3f})')
plt.tight_layout()
plt.show()

In [ ]:
model.save(f"ghgsat_cnn_{method}.h5")
print("Model saved.")